# Preprocesameinto

In [1]:
import pandas as pd
import numpy as np
import joblib
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from preprocessing import AmesFeatureEngineer, TARGET

df = pd.read_csv('data/train.csv')  # ajustar ruta según estructura del repo
df.shape

(1168, 81)

## Separación en train y validation

In [2]:
X = df.drop(columns=[TARGET, 'Id'])
y = np.log1p(df[TARGET])  #transformación log 

X_train, X_val, y_train, y_val = train_test_split(
    X, y, test_size=0.2, random_state=42
)

print(f"Train: {X_train.shape}, Val: {X_val.shape}")

Train: (934, 79), Val: (234, 79)


In [3]:
feat_eng = AmesFeatureEngineer()
X_train_fe = feat_eng.fit_transform(X_train)
X_val_fe = feat_eng.transform(X_val)

nominal_cols = X_train_fe.select_dtypes(include='object').columns.tolist()
numeric_cols = X_train_fe.select_dtypes(include=['int64', 'float64']).columns.tolist()

print(f"Nominales (one-hot): {len(nominal_cols)}")
print(f"Numéricas u ordinales (escaladas): {len(numeric_cols)}")

Nominales (one-hot): 23
Numéricas u ordinales (escaladas): 56


In [4]:
preprocessor = ColumnTransformer(transformers=[
    ('num', Pipeline([
        ('imputer', SimpleImputer(strategy='median')),
        ('scaler', StandardScaler())
    ]), numeric_cols),
    ('cat', Pipeline([
        ('imputer', SimpleImputer(strategy='most_frequent')),
        ('onehot', OneHotEncoder(handle_unknown='ignore', sparse_output=False))
    ]), nominal_cols),
])

X_train_processed = preprocessor.fit_transform(X_train_fe)
X_val_processed = preprocessor.transform(X_val_fe)

print(f"X_train_processed shape: {X_train_processed.shape}")
print(f"X_val_processed shape: {X_val_processed.shape}")

X_train_processed shape: (934, 223)
X_val_processed shape: (234, 223)


# Guardar pipeline completo

In [6]:
joblib.dump({
    'feature_engineer': feat_eng,
    'preprocessor': preprocessor,
    'numeric_cols': numeric_cols,
    'nominal_cols': nominal_cols,
}, 'models/preprocessing_pipeline.joblib') 

np.save('data/X_train_processed.npy', X_train_processed)
np.save('data/X_val_processed.npy', X_val_processed)
np.save('data/y_train.npy', y_train.values)
np.save('data/y_val.npy', y_val.values)

np.save('data/X_val_index.npy', X_val.index.values)
print("Pipeline y datos procesados guardados correctamente")

Pipeline y datos procesados guardados correctamente
